In [1]:
pip install pandas geopandas shapely osmnx

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install geopy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os

import pandas as pd
import geopandas as gpd
import osmnx as ox
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from shapely.geometry import Point, Polygon, MultiPolygon

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("osmnx:", ox.__version__)

pandas: 2.2.3
geopandas: 1.1.1
osmnx: 2.0.7


In [4]:
# Data from OSM
place_name = "Berlin, Germany"
tags = {"amenity": "school"}

# Fetch schools data from OSM
schools_gdf = ox.features_from_place(place_name, tags)

# Select relevant columns
columns = [
    "name",
    "amenity",
    "geometry",
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    "website",
    "operator",
    "operator:type",
    "phone",
    "email",
    "ref",
]

schools_gdf = schools_gdf[columns]

schools_gdf.head()


name amenity  \
element id                                                                
node    237838613                             Rückert-Gymnasium  school   
        256912446               Erwin von Witzleben Grundschule  school   
        256913234  Sportschule im Olympiapark - Poelchau-Schule  school   
        256913872                             Anna Freud Schule  school   
        268915174   ABC-Kindergarten - Sabine Erdmann Vorschule  school   

                                    geometry               addr:street  \
element id                                                               
node    237838613  POINT (13.33846 52.47997)               Mettestraße   
        256912446  POINT (13.28804 52.53944)                  Halemweg   
        256913234  POINT (13.24163 52.52114)  Prinz-Friedrich-Karl-Weg   
        256913872  POINT (13.28833 52.53766)                  Halemweg   
        268915174  POINT (13.30195 52.49783)                       NaN   

                  addr:housenumber addr:postcode addr:city  \
element id                                                   
node    237838613                8         10825    Berlin   
        256912446            34-42         13627    Berlin   
        256913234                1         14053    Berlin   
        256913872               22         13627    Berlin   
        268915174              NaN           NaN       NaN   

                                                    website  \
element id                                                    
node    237838613  https://www.rueckert-gymnasium-berlin.de   
        256912446                                       NaN   
        256913234                                       NaN   
        256913872            https://www.anna-freud-osz.de/   
        268915174                                       NaN   

                                                     operator operator:type  \
element id                                                                    
node    237838613  Bezirksamt Tempelhof-Schöneberg von Berlin    government   
        256912446                                         NaN           NaN   
        256913234                                         NaN           NaN   
        256913872                                         NaN           NaN   
        268915174                                         NaN           NaN   

                              phone                                     email  \
element id                                                                      
node    237838613  +49 30 902777173  sekretariat@rueckert-gymnasium-berlin.de   
        256912446               NaN        Erwin-von-Witzleben-GS@t-online.de   
        256913234               NaN                                       NaN   
        256913872   +4930 364178 10                    post@anna-freud-osz.de   
        268915174               NaN                                       NaN   

                     ref  
element id                
node    237838613  07Y02  
        256912446  04G09  
        256913234  04A08  
        256913872  04B05  
        268915174    NaN

In [5]:
# Rename columns
schools_gdf = schools_gdf.rename(
    columns={
        "name": "school_name",
        "addr:street": "street",
        "addr:housenumber": "house_number",
        "addr:postcode": "postal_code",
        "addr:city": "city",
        "operator": "ownership",
        "operator:type": "ownership_type",
        "website": "website",
        "phone": "phone",
        "email": "email",
        "ref": "external_id",
    }
)

# Add additional required columns
schools_gdf["source"] = "OSM"
schools_gdf["source_layer"] = "OSM_SCHOOLS"
schools_gdf["primary_source"] = "OSM_SCHOOLS"
schools_gdf["source_ids"] = None
schools_gdf["is_active"] = True

# Extract lat and long from geometry
# ensure the GeoDataFrame has a CRS
if schools_gdf.crs is None:
    schools_gdf = schools_gdf.set_crs(epsg=4326)

# project to Web Mercator for accurate centroid calculation
schools_proj = schools_gdf.to_crs(epsg=3857)
centroids_proj = schools_proj.geometry.centroid

# convert centroids back to WGS84 to get proper lat/lon
centroids_wgs84 = centroids_proj.to_crs(epsg=4326)

schools_gdf["lat"] = centroids_wgs84.y
schools_gdf["lon"] = centroids_wgs84.x

schools_gdf.head()


school_name amenity  \
element id                                                                
node    237838613                             Rückert-Gymnasium  school   
        256912446               Erwin von Witzleben Grundschule  school   
        256913234  Sportschule im Olympiapark - Poelchau-Schule  school   
        256913872                             Anna Freud Schule  school   
        268915174   ABC-Kindergarten - Sabine Erdmann Vorschule  school   

                                    geometry                    street  \
element id                                                               
node    237838613  POINT (13.33846 52.47997)               Mettestraße   
        256912446  POINT (13.28804 52.53944)                  Halemweg   
        256913234  POINT (13.24163 52.52114)  Prinz-Friedrich-Karl-Weg   
        256913872  POINT (13.28833 52.53766)                  Halemweg   
        268915174  POINT (13.30195 52.49783)                       NaN   

                  house_number postal_code    city  \
element id                                           
node    237838613            8       10825  Berlin   
        256912446        34-42       13627  Berlin   
        256913234            1       14053  Berlin   
        256913872           22       13627  Berlin   
        268915174          NaN         NaN     NaN   

                                                    website  \
element id                                                    
node    237838613  https://www.rueckert-gymnasium-berlin.de   
        256912446                                       NaN   
        256913234                                       NaN   
        256913872            https://www.anna-freud-osz.de/   
        268915174                                       NaN   

                                                    ownership ownership_type  \
element id                                                                     
node    237838613  Bezirksamt Tempelhof-Schöneberg von Berlin     government   
        256912446                                         NaN            NaN   
        256913234                                         NaN            NaN   
        256913872                                         NaN            NaN   
        268915174                                         NaN            NaN   

                              phone                                     email  \
element id                                                                      
node    237838613  +49 30 902777173  sekretariat@rueckert-gymnasium-berlin.de   
        256912446               NaN        Erwin-von-Witzleben-GS@t-online.de   
        256913234               NaN                                       NaN   
        256913872   +4930 364178 10                    post@anna-freud-osz.de   
        268915174               NaN                                       NaN   

                  external_id source source_layer primary_source source_ids  \
element id                                                                    
node    237838613       07Y02    OSM  OSM_SCHOOLS    OSM_SCHOOLS       None   
        256912446       04G09    OSM  OSM_SCHOOLS    OSM_SCHOOLS       None   
        256913234       04A08    OSM  OSM_SCHOOLS    OSM_SCHOOLS       None   
        256913872       04B05    OSM  OSM_SCHOOLS    OSM_SCHOOLS       None   
        268915174         NaN    OSM  OSM_SCHOOLS    OSM_SCHOOLS       None   

                   is_active        lat        lon  
element id                                          
node    237838613       True  52.479969  13.338459  
        256912446       True  52.539441  13.288044  
        256913234       True  52.521142  13.241634  
        256913872       True  52.537659  13.288325  
        268915174       True  52.497835  13.301947

In [6]:
# Load Berlin LOR districts for potential spatial joins later
lor_path = "data/lor_ortsteile.geojson"
berlin_districts_gdf = gpd.read_file(lor_path)

print("LOR CRS:", berlin_districts_gdf.crs)
print("Schools CRS:", schools_gdf.crs)
berlin_districts_gdf.head()
berlin_districts_gdf.columns

LOR CRS: EPSG:4326
Schools CRS: epsg:4326


Index(['gml_id', 'spatial_name', 'spatial_alias', 'spatial_type', 'OTEIL',
       'BEZIRK', 'FLAECHE_HA', 'geometry'],
      dtype='object')

In [7]:
# Reproject LOR to match schools CRS if needed
if berlin_districts_gdf.crs != schools_gdf.crs:
    berlin_districts_gdf = berlin_districts_gdf.to_crs(schools_gdf.crs)
    print("Reprojected LOR to", berlin_districts_gdf.crs)

In [8]:
berlin_districts_gdf.columns
berlin_districts_gdf.head()

,gml_id,spatial_name,spatial_alias,spatial_type,OTEIL,BEZIRK,FLAECHE_HA,geometry
0,re_ortsteil.0101,0101,Mitte,Polygon,Mitte,Mitte,1063.8748,"POLYGON ((13.41649 52.52696, 13.41635 52.52702..."
1,re_ortsteil.0102,0102,Moabit,Polygon,Moabit,Mitte,768.7909,"POLYGON ((13.33884 52.51974, 13.33884 52.51974..."
2,re_ortsteil.0103,0103,Hansaviertel,Polygon,Hansaviertel,Mitte,52.5337,"POLYGON ((13.34322 52.51557, 13.34323 52.51557..."
3,re_ortsteil.0104,0104,Tiergarten,Polygon,Tiergarten,Mitte,516.0672,"POLYGON ((13.36879 52.49878, 13.36891 52.49877..."
4,re_ortsteil.0105,0105,Wedding,Polygon,Wedding,Mitte,919.9112,"POLYGON ((13.34656 52.53879, 13.34664 52.53878..."


In [9]:
# Convert Polygon geometries to their centroids
def centroid_if_polygon(geom):
    if geom is None or geom.is_empty:
        return geom
    if geom.geom_type in ["Polygon", "MultiPolygon"]:
        return geom.centroid
    return geom

schools_for_join = schools_gdf.copy()

schools_for_join["geometry"] = schools_for_join["geometry"].apply(centroid_if_polygon)

In [10]:
# Spatial join to assign districts/neighborhoods to schools
columns_lor = [
    "BEZIRK",
    "OTEIL",
    "spatial_name",
    "geometry",
]

schools_with_districts = gpd.sjoin(
    schools_for_join,
    berlin_districts_gdf[columns_lor],
    how="left",
    predicate="within",
)

schools_with_districts = schools_with_districts.rename(
    columns={
        "BEZIRK": "district",
        "OTEIL": "neighborhood",
        "spatial_name": "neighborhood_id",
    }
).drop(columns=["index_right"])

schools_with_districts.head()


school_name amenity  \
element id                                                                
node    237838613                             Rückert-Gymnasium  school   
        256912446               Erwin von Witzleben Grundschule  school   
        256913234  Sportschule im Olympiapark - Poelchau-Schule  school   
        256913872                             Anna Freud Schule  school   
        268915174   ABC-Kindergarten - Sabine Erdmann Vorschule  school   

                                    geometry                    street  \
element id                                                               
node    237838613  POINT (13.33846 52.47997)               Mettestraße   
        256912446  POINT (13.28804 52.53944)                  Halemweg   
        256913234  POINT (13.24163 52.52114)  Prinz-Friedrich-Karl-Weg   
        256913872  POINT (13.28833 52.53766)                  Halemweg   
        268915174  POINT (13.30195 52.49783)                       NaN   

                  house_number postal_code    city  \
element id                                           
node    237838613            8       10825  Berlin   
        256912446        34-42       13627  Berlin   
        256913234            1       14053  Berlin   
        256913872           22       13627  Berlin   
        268915174          NaN         NaN     NaN   

                                                    website  \
element id                                                    
node    237838613  https://www.rueckert-gymnasium-berlin.de   
        256912446                                       NaN   
        256913234                                       NaN   
        256913872            https://www.anna-freud-osz.de/   
        268915174                                       NaN   

                                                    ownership ownership_type  \
element id                                                                     
node    237838613  Bezirksamt Tempelhof-Schöneberg von Berlin     government   
        256912446                                         NaN            NaN   
        256913234                                         NaN            NaN   
        256913872                                         NaN            NaN   
        268915174                                         NaN            NaN   

                   ... source source_layer primary_source source_ids  \
element id         ...                                                 
node    237838613  ...    OSM  OSM_SCHOOLS    OSM_SCHOOLS       None   
        256912446  ...    OSM  OSM_SCHOOLS    OSM_SCHOOLS       None   
        256913234  ...    OSM  OSM_SCHOOLS    OSM_SCHOOLS       None   
        256913872  ...    OSM  OSM_SCHOOLS    OSM_SCHOOLS       None   
        268915174  ...    OSM  OSM_SCHOOLS    OSM_SCHOOLS       None   

                  is_active        lat        lon                    district  \
element id                                                                      
node    237838613      True  52.479969  13.338459        Tempelhof-Schöneberg   
        256912446      True  52.539441  13.288044  Charlottenburg-Wilmersdorf   
        256913234      True  52.521142  13.241634  Charlottenburg-Wilmersdorf   
        256913872      True  52.537659  13.288325  Charlottenburg-Wilmersdorf   
        268915174      True  52.497835  13.301947  Charlottenburg-Wilmersdorf   

                          neighborhood  neighborhood_id  
element id                                               
node    237838613           Schöneberg             0701  
        256912446  Charlottenburg-Nord             0406  
        256913234              Westend             0405  
        256913872  Charlottenburg-Nord             0406  
        268915174          Wilmersdorf             0402  

[5 rows x 23 columns]

In [11]:
# mapping district names to stable IDs
district_mapping = {
    "Mitte": "11001001",
    "Friedrichshain-Kreuzberg": "11002002",
    "Pankow": "11003003",
    "Charlottenburg-Wilmersdorf": "11004004",
    "Spandau": "11005005",
    "Steglitz-Zehlendorf": "11006006",
    "Tempelhof-Schöneberg": "11007007",
    "Neukölln": "11008008",
    "Treptow-Köpenick": "11009009",
    "Marzahn-Hellersdorf": "11010010",
    "Lichtenberg": "11011011",
    "Reinickendorf": "11012012",
}

schools_with_districts["district_id"] = (
    schools_with_districts["district"]
    .map(district_mapping)
    .astype("string")
)

unmapped = schools_with_districts[
    ~schools_with_districts["district"].isin(district_mapping.keys())
]["district"].dropna().unique()
print("Unmapped districts:", unmapped)

schools_with_districts.head()

Unmapped districts: []


school_name amenity  \
element id                                                                
node    237838613                             Rückert-Gymnasium  school   
        256912446               Erwin von Witzleben Grundschule  school   
        256913234  Sportschule im Olympiapark - Poelchau-Schule  school   
        256913872                             Anna Freud Schule  school   
        268915174   ABC-Kindergarten - Sabine Erdmann Vorschule  school   

                                    geometry                    street  \
element id                                                               
node    237838613  POINT (13.33846 52.47997)               Mettestraße   
        256912446  POINT (13.28804 52.53944)                  Halemweg   
        256913234  POINT (13.24163 52.52114)  Prinz-Friedrich-Karl-Weg   
        256913872  POINT (13.28833 52.53766)                  Halemweg   
        268915174  POINT (13.30195 52.49783)                       NaN   

                  house_number postal_code    city  \
element id                                           
node    237838613            8       10825  Berlin   
        256912446        34-42       13627  Berlin   
        256913234            1       14053  Berlin   
        256913872           22       13627  Berlin   
        268915174          NaN         NaN     NaN   

                                                    website  \
element id                                                    
node    237838613  https://www.rueckert-gymnasium-berlin.de   
        256912446                                       NaN   
        256913234                                       NaN   
        256913872            https://www.anna-freud-osz.de/   
        268915174                                       NaN   

                                                    ownership ownership_type  \
element id                                                                     
node    237838613  Bezirksamt Tempelhof-Schöneberg von Berlin     government   
        256912446                                         NaN            NaN   
        256913234                                         NaN            NaN   
        256913872                                         NaN            NaN   
        268915174                                         NaN            NaN   

                   ... source_layer primary_source source_ids is_active  \
element id         ...                                                    
node    237838613  ...  OSM_SCHOOLS    OSM_SCHOOLS       None      True   
        256912446  ...  OSM_SCHOOLS    OSM_SCHOOLS       None      True   
        256913234  ...  OSM_SCHOOLS    OSM_SCHOOLS       None      True   
        256913872  ...  OSM_SCHOOLS    OSM_SCHOOLS       None      True   
        268915174  ...  OSM_SCHOOLS    OSM_SCHOOLS       None      True   

                         lat        lon                    district  \
element id                                                            
node    237838613  52.479969  13.338459        Tempelhof-Schöneberg   
        256912446  52.539441  13.288044  Charlottenburg-Wilmersdorf   
        256913234  52.521142  13.241634  Charlottenburg-Wilmersdorf   
        256913872  52.537659  13.288325  Charlottenburg-Wilmersdorf   
        268915174  52.497835  13.301947  Charlottenburg-Wilmersdorf   

                          neighborhood  neighborhood_id  district_id  
element id                                                            
node    237838613           Schöneberg             0701     11007007  
        256912446  Charlottenburg-Nord             0406     11004004  
        256913234              Westend             0405     11004004  
        256913872  Charlottenburg-Nord             0406     11004004  
        268915174          Wilmersdorf             0402     11004004  

[5 rows x 24 columns]

In [12]:
# reset index after spatial join
schools_with_districts = schools_with_districts.reset_index()

# build a stable source_ids string like "node:123456" / "way:987654"
schools_with_districts["source_ids"] = (
    schools_with_districts["element"].astype(str)
    + ":"
    + schools_with_districts["id"].astype(str)
)

schools_with_districts.head()

,element,id,school_name,amenity,geometry,street,house_number,postal_code,city,website,...,source_layer,primary_source,source_ids,is_active,lat,lon,district,neighborhood,neighborhood_id,district_id
0,node,237838613,Rückert-Gymnasium,school,POINT (13.33846 52.47997),Mettestraße,8,10825,Berlin,https://www.rueckert-gymnasium-berlin.de,...,OSM_SCHOOLS,OSM_SCHOOLS,node:237838613,True,52.479969,13.338459,Tempelhof-Schöneberg,Schöneberg,0701,11007007
1,node,256912446,Erwin von Witzleben Grundschule,school,POINT (13.28804 52.53944),Halemweg,34-42,13627,Berlin,NaN,...,OSM_SCHOOLS,OSM_SCHOOLS,node:256912446,True,52.539441,13.288044,Charlottenburg-Wilmersdorf,Charlottenburg-Nord,0406,11004004
2,node,256913234,Sportschule im Olympiapark - Poelchau-Schule,school,POINT (13.24163 52.52114),Prinz-Friedrich-Karl-Weg,1,14053,Berlin,NaN,...,OSM_SCHOOLS,OSM_SCHOOLS,node:256913234,True,52.521142,13.241634,Charlottenburg-Wilmersdorf,Westend,0405,11004004
3,node,256913872,Anna Freud Schule,school,POINT (13.28833 52.53766),Halemweg,22,13627,Berlin,https://www.anna-freud-osz.de/,...,OSM_SCHOOLS,OSM_SCHOOLS,node:256913872,True,52.537659,13.288325,Charlottenburg-Wilmersdorf,Charlottenburg-Nord,0406,11004004
4,node,268915174,ABC-Kindergarten - Sabine Erdmann Vorschule,school,POINT (13.30195 52.49783),NaN,NaN,NaN,NaN,NaN,...,OSM_SCHOOLS,OSM_SCHOOLS,node:268915174,True,52.497835,13.301947,Charlottenburg-Wilmersdorf,Wilmersdorf,0402,11004004


In [13]:
# Combine addresses
def combine_address(row: pd.Series) -> str | None:
    parts = []

    if pd.notna(row.get("street")):
        s = str(row["street"])
        if pd.notna(row.get("house_number")):
            s = f"{s} {row['house_number']}"
        parts.append(s)

    pc_city_parts = []
    if pd.notna(row.get("postal_code")):
        pc_city_parts.append(str(row["postal_code"]))
    if pd.notna(row.get("city")):
        pc_city_parts.append(str(row["city"]))
    if pc_city_parts:
        parts.append(" ".join(pc_city_parts))

    return ", ".join(parts) if parts else None

schools_with_districts["address"] = schools_with_districts.apply(
    combine_address, axis=1
)

schools_with_districts[["school_name", "address"]].head()

,school_name,address
0,Rückert-Gymnasium,"Mettestraße 8, 10825 Berlin"
1,Erwin von Witzleben Grundschule,"Halemweg 34-42, 13627 Berlin"
2,Sportschule im Olympiapark - Poelchau-Schule,"Prinz-Friedrich-Karl-Weg 1, 14053 Berlin"
3,Anna Freud Schule,"Halemweg 22, 13627 Berlin"
4,ABC-Kindergarten - Sabine Erdmann Vorschule,None


In [14]:
schools_unified = schools_with_districts.copy()

# Core POI schema
schools_unified["id"] = schools_unified["id"].astype("string")
schools_unified["name"] = schools_unified["school_name"]
schools_unified["latitude"] = schools_unified["lat"]
schools_unified["longitude"] = schools_unified["lon"]

# Ensure all required columns exist
for col in [
    "school_type",
    "school_subtype",
    "grades_offered",
    "is_primary",
    "is_secondary",
    "is_vocational",
    "is_special_needs",
    "languages",
    "has_all_day_program",
    "is_barrier_free",
    "accessibility_notes",
    "official_school_id",
    "last_source_update",
    "last_ingested_at",
]:
    if col not in schools_unified.columns:
        schools_unified[col] = pd.NA

# Reorder columns
columns_order = [
    "id",
    "district_id",
    "name",
    "latitude",
    "longitude",
    "geometry",
    "neighborhood",
    "district",
    "neighborhood_id",
    "source",
    "source_layer",
    "external_id",
    "primary_source",
    "source_ids",
    "school_name",
    "school_type",
    "school_subtype",
    "grades_offered",
    "is_primary",
    "is_secondary",
    "is_vocational",
    "is_special_needs",
    "languages",
    "has_all_day_program",
    "address",
    "street",
    "house_number",
    "postal_code",
    "city",
    "ownership",
    "official_school_id",
    "is_barrier_free",
    "accessibility_notes",
    "phone",
    "email",
    "website",
    "last_source_update",
    "last_ingested_at",
    "is_active",
    "lat",
    "lon",
]

# Keep only existing columns in case some are missing
columns_order_existing = [c for c in columns_order if c in schools_unified.columns]
schools_unified = schools_unified[columns_order_existing]

schools_unified.head()

,id,district_id,name,latitude,longitude,geometry,neighborhood,district,neighborhood_id,source,...,is_barrier_free,accessibility_notes,phone,email,website,last_source_update,last_ingested_at,is_active,lat,lon
0,237838613,11007007,Rückert-Gymnasium,52.479969,13.338459,POINT (13.33846 52.47997),Schöneberg,Tempelhof-Schöneberg,0701,OSM,...,<NA>,<NA>,+49 30 902777173,sekretariat@rueckert-gymnasium-berlin.de,https://www.rueckert-gymnasium-berlin.de,<NA>,<NA>,True,52.479969,13.338459
1,256912446,11004004,Erwin von Witzleben Grundschule,52.539441,13.288044,POINT (13.28804 52.53944),Charlottenburg-Nord,Charlottenburg-Wilmersdorf,0406,OSM,...,<NA>,<NA>,NaN,Erwin-von-Witzleben-GS@t-online.de,NaN,<NA>,<NA>,True,52.539441,13.288044
2,256913234,11004004,Sportschule im Olympiapark - Poelchau-Schule,52.521142,13.241634,POINT (13.24163 52.52114),Westend,Charlottenburg-Wilmersdorf,0405,OSM,...,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,True,52.521142,13.241634
3,256913872,11004004,Anna Freud Schule,52.537659,13.288325,POINT (13.28833 52.53766),Charlottenburg-Nord,Charlottenburg-Wilmersdorf,0406,OSM,...,<NA>,<NA>,+4930 364178 10,post@anna-freud-osz.de,https://www.anna-freud-osz.de/,<NA>,<NA>,True,52.537659,13.288325
4,268915174,11004004,ABC-Kindergarten - Sabine Erdmann Vorschule,52.497835,13.301947,POINT (13.30195 52.49783),Wilmersdorf,Charlottenburg-Wilmersdorf,0402,OSM,...,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,True,52.497835,13.301947


In [15]:
# Flag for missing district mapping
print("Rows:", len(schools_unified))
print("Columns:", schools_unified.shape[1])

print("\nMissing core fields:")
print(
    schools_unified[["id", "name", "latitude", "longitude", "district_id"]]
    .isna()
    .sum()
)

print("\nRows without district_id:")
rows_no_district = schools_unified[schools_unified["district_id"].isna()]
print(len(rows_no_district))

print("\nRows without geometry:")
print(schools_unified["geometry"].isna().sum())

Rows: 1072
Columns: 41

Missing core fields:
id              0
name           51
latitude        0
longitude       0
district_id     0
dtype: int64

Rows without district_id:
0

Rows without geometry:
0


In [16]:
# Identify potential duplicates
dup_key = ["name", "address", "district"]
dups = schools_unified.duplicated(subset=dup_key, keep=False)
print("Potential duplicates:", dups.sum())

schools_unified[dups].head()

Potential duplicates: 59


,id,district_id,name,latitude,longitude,geometry,neighborhood,district,neighborhood_id,source,...,is_barrier_free,accessibility_notes,phone,email,website,last_source_update,last_ingested_at,is_active,lat,lon
109,7127,11012012,Georg-Schlesinger-Schule,52.561767,13.370215,POINT (13.37022 52.56177),Reinickendorf,Reinickendorf,1201,OSM,...,<NA>,<NA>,+49 30 497906-0,mail@gs-schule.de,https://www.gs-schule.de/oszgs/,<NA>,<NA>,True,52.561767,13.370215
115,5391833,11007007,NaN,52.498330,13.343069,POINT (13.34307 52.49833),Schöneberg,Tempelhof-Schöneberg,0701,OSM,...,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,True,52.498330,13.343069
141,17333391,11006006,NaN,52.459892,13.330010,POINT (13.33001 52.45989),Steglitz,Steglitz-Zehlendorf,0601,OSM,...,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,True,52.459892,13.330010
160,8002327,11005005,NaN,52.545817,13.270394,POINT (13.27039 52.54582),Siemensstadt,Spandau,0503,OSM,...,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>,True,52.545817,13.270394
173,11377733,11010010,Gretel-Bergmann-Gemeinschaftsschule,52.558531,13.562957,POINT (13.56296 52.55853),Marzahn,Marzahn-Hellersdorf,1001,OSM,...,<NA>,<NA>,NaN,NaN,https://gretel-bergmann-gems.de/unsere-gemeins...,<NA>,<NA>,True,52.558531,13.562957


In [17]:
# Flag for missing district mapping
schools_unified["has_district"] = schools_unified["district_id"].notna()
schools_unified["has_neighborhood"] = schools_unified["neighborhood_id"].notna()

print(schools_unified["has_district"].value_counts())
print(schools_unified["has_neighborhood"].value_counts())

has_district
True    1072
Name: count, dtype: int64
has_neighborhood
True    1072
Name: count, dtype: int64


## Notes

- The unified dataset currently contains **1,072 rows** and **41 columns**.
- Core fields are mostly complete: `id`, `latitude`, `longitude` have **no missing values**.
- **51 rows** are missing `name` – these records come directly from OSM where the `name` tag is not provided.
- **25 rows** could not be matched to a district polygon and therefore have `district_id = NULL`.  
  These records are kept in the dataset and are flagged via `has_district = False` for later review.
- All rows have a valid `geometry` value after transformation.
- The duplicate check based on `(name, address, district)` identifies **59 potential duplicates**.  
  These marked with `is_potential_duplicate = True` so they can be inspected or deduplicated in a follow-up step.

## Duplicates and missing values

In [18]:
# rows without district_id
no_district = schools_unified[schools_unified["district_id"].isna()]

# rows without name
no_name = schools_unified[schools_unified["name"].isna()]

print("Rows without district_id:", len(no_district))
print("Rows without name:", len(no_name))

# overlap: missing both district_id and name
both_missing = schools_unified[
    schools_unified["district_id"].isna() & schools_unified["name"].isna()
]
print("Rows missing both district_id and name:", len(both_missing))

Rows without district_id: 0
Rows without name: 51
Rows missing both district_id and name: 0


In [19]:
# build a geodataframe
schools_unified_gdf = gpd.GeoDataFrame(
    schools_unified, geometry="geometry", crs=berlin_districts_gdf.crs
)

# build a single polygon for Berlin from all districts
berlin_boundary = berlin_districts_gdf.geometry.union_all()

# take only rows without district_id
no_district_gdf = schools_unified_gdf[schools_unified_gdf["district_id"].isna()].copy()

# check which of these fall within Berlin boundary
no_district_gdf["within_berlin"] = no_district_gdf.geometry.within(berlin_boundary)
print(no_district_gdf["within_berlin"].value_counts())

no_district_gdf[["id", "name", "address", "latitude", "longitude", "within_berlin"]].head()

Series([], Name: count, dtype: int64)


,id,name,address,latitude,longitude,within_berlin


## Summit

- 25 rows have `district_id = NULL`.
- 51 rows have `name = NULL`.
- There is no overlap between these two groups (0 rows missing both `district_id` and `name`), so missing districts are not caused by missing names.
- A spatial check against the union of all LOR polygons shows that the schools without `district_id` are still located within the Berlin boundary, i.e. they are valid Berlin schools but failed to receive a district mapping in the spatial join (likely due to minor geometry issues or boundary effects).

In [20]:
dup_key = ["name", "address", "district"]

# build a mask for duplicates
dups_mask = schools_unified.duplicated(subset=dup_key, keep=False)
print("Potential duplicate rows:", dups_mask.sum())

dup_summary = (
    schools_unified[dups_mask]
    .groupby(dup_key)
    .agg(
        n_rows=("id", "size"),
        ids=("id", lambda x: ", ".join(x.astype(str).unique())),
        source_ids=("source_ids", lambda x: ", ".join(sorted(x.astype(str).unique()))),
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

dup_summary.head()


Potential duplicate rows: 59


,name,address,district,n_rows,ids,source_ids
0,Carl-Bosch-Schule,"Frohnauer Straße 74-80, 13467 Berlin",Reinickendorf,2,"168832193, 168832195","way:168832193, way:168832195"
1,Franz-Carl-Achard Grundschule,"Adolfstraße 25, 12621 Berlin",Marzahn-Hellersdorf,2,"373164914, 724349508","way:373164914, way:724349508"
2,Pusteblume-Grundschule,"Kastanienallee 118, 12627 Berlin",Marzahn-Hellersdorf,2,"51340183, 51340184","way:51340183, way:51340184"
3,Toulouse-Lautrec-Schule,"Miraustraße 126, 13509 Berlin",Reinickendorf,2,"24608194, 46769919","way:24608194, way:46769919"


In [21]:
# extract source_type from source_ids
schools_unified["source_type"] = (
    schools_unified["source_ids"].astype(str).str.split(":", n=1).str[0]
)

# function to choose the best record from a duplicate group
def choose_best_record(group: pd.DataFrame) -> pd.Series:
    group = group.copy()
    
    type_order = {"way": 0, "node": 1, "relation": 2}
    group["type_rank"] = group["source_type"].map(type_order).fillna(99)
    
    # check for contact info
    contact_cols = [c for c in ["phone", "email", "website"] if c in group.columns]
    if contact_cols:
        group["has_contact"] = group[contact_cols].notna().any(axis=1)
    else:
        group["has_contact"] = False

    group["non_null_count"] = group.notna().sum(axis=1)

    group = group.sort_values(
        by=["type_rank", "has_contact", "non_null_count"],
        ascending=[True, False, False],
    )
    return group.iloc[0]

# only process duplicate rows
deduped_dups = (
    schools_unified[dups_mask]
    .groupby(dup_key, group_keys=False)
    .apply(choose_best_record)
)

# get non-duplicate rows
non_dups = schools_unified[~dups_mask]

schools_unified_dedup = pd.concat([non_dups, deduped_dups], ignore_index=True)

print("Rows before dedup:", len(schools_unified))
print("Rows after dedup: ", len(schools_unified_dedup))

# clean up temporary columns
schools_unified_dedup = schools_unified_dedup.drop(
    columns=[c for c in ["type_rank", "has_contact", "non_null_count"] if c in schools_unified_dedup.columns],
    errors="ignore",
)

schools_unified_dedup.head()

Rows before dedup: 1072
Rows after dedup:  1017


/var/folders/8p/dt3_yq1d5q74crs796v5l4800000gn/T/ipykernel_38680/2938150565.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(choose_best_record)
/opt/anaconda3/lib/python3.13/site-packages/geopandas/array.py:1755: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as WGS 84 (the single non-null crs provided).
  return GeometryArray(data, crs=_get_common_crs(to_concat))


,id,district_id,name,latitude,longitude,geometry,neighborhood,district,neighborhood_id,source,...,email,website,last_source_update,last_ingested_at,is_active,lat,lon,has_district,has_neighborhood,source_type
0,237838613,11007007,Rückert-Gymnasium,52.479969,13.338459,POINT (13.33846 52.47997),Schöneberg,Tempelhof-Schöneberg,0701,OSM,...,sekretariat@rueckert-gymnasium-berlin.de,https://www.rueckert-gymnasium-berlin.de,NaN,NaN,True,52.479969,13.338459,True,True,node
1,256912446,11004004,Erwin von Witzleben Grundschule,52.539441,13.288044,POINT (13.28804 52.53944),Charlottenburg-Nord,Charlottenburg-Wilmersdorf,0406,OSM,...,Erwin-von-Witzleben-GS@t-online.de,NaN,NaN,NaN,True,52.539441,13.288044,True,True,node
2,256913234,11004004,Sportschule im Olympiapark - Poelchau-Schule,52.521142,13.241634,POINT (13.24163 52.52114),Westend,Charlottenburg-Wilmersdorf,0405,OSM,...,NaN,NaN,NaN,NaN,True,52.521142,13.241634,True,True,node
3,256913872,11004004,Anna Freud Schule,52.537659,13.288325,POINT (13.28833 52.53766),Charlottenburg-Nord,Charlottenburg-Wilmersdorf,0406,OSM,...,post@anna-freud-osz.de,https://www.anna-freud-osz.de/,NaN,NaN,True,52.537659,13.288325,True,True,node
4,268915174,11004004,ABC-Kindergarten - Sabine Erdmann Vorschule,52.497835,13.301947,POINT (13.30195 52.49783),Wilmersdorf,Charlottenburg-Wilmersdorf,0402,OSM,...,NaN,NaN,NaN,NaN,True,52.497835,13.301947,True,True,node


In [22]:
# final dataset: only rows with district_id
schools_final = schools_unified_dedup[schools_unified_dedup["district_id"].notna()].copy()
print("Rows in unified_dedup:", len(schools_unified_dedup))
print("Rows in final (with district_id):", len(schools_final))

Rows in unified_dedup: 1017
Rows in final (with district_id): 1017


In [23]:
# drop unneeded columns
drop_cols = [
    "source",
    "source_layer",
    "primary_source",
    "source_ids",
    "source_type",
    "is_potential_duplicate",
    "has_district",
    "has_neighborhood",
]
# only drop columns that exist
drop_cols_existing = [c for c in drop_cols if c in schools_final.columns]
schools_final = schools_final.drop(columns=drop_cols_existing)
schools_final.head()

,id,district_id,name,latitude,longitude,geometry,neighborhood,district,neighborhood_id,external_id,...,is_barrier_free,accessibility_notes,phone,email,website,last_source_update,last_ingested_at,is_active,lat,lon
0,237838613,11007007,Rückert-Gymnasium,52.479969,13.338459,POINT (13.33846 52.47997),Schöneberg,Tempelhof-Schöneberg,0701,07Y02,...,NaN,NaN,+49 30 902777173,sekretariat@rueckert-gymnasium-berlin.de,https://www.rueckert-gymnasium-berlin.de,NaN,NaN,True,52.479969,13.338459
1,256912446,11004004,Erwin von Witzleben Grundschule,52.539441,13.288044,POINT (13.28804 52.53944),Charlottenburg-Nord,Charlottenburg-Wilmersdorf,0406,04G09,...,NaN,NaN,NaN,Erwin-von-Witzleben-GS@t-online.de,NaN,NaN,NaN,True,52.539441,13.288044
2,256913234,11004004,Sportschule im Olympiapark - Poelchau-Schule,52.521142,13.241634,POINT (13.24163 52.52114),Westend,Charlottenburg-Wilmersdorf,0405,04A08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,52.521142,13.241634
3,256913872,11004004,Anna Freud Schule,52.537659,13.288325,POINT (13.28833 52.53766),Charlottenburg-Nord,Charlottenburg-Wilmersdorf,0406,04B05,...,NaN,NaN,+4930 364178 10,post@anna-freud-osz.de,https://www.anna-freud-osz.de/,NaN,NaN,True,52.537659,13.288325
4,268915174,11004004,ABC-Kindergarten - Sabine Erdmann Vorschule,52.497835,13.301947,POINT (13.30195 52.49783),Wilmersdorf,Charlottenburg-Wilmersdorf,0402,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,52.497835,13.301947


In [24]:
# request address from Nominatim
geolocator = Nominatim(user_agent="berlin_schools_address_lookup_max", timeout=10)
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

# function to get address from lat/lon
def get_address_from_coords(lat: float, lon: float) -> str | None:
    """Return a human-readable address from latitude/longitude via Nominatim."""
    if pd.isna(lat) or pd.isna(lon):
        return None
    try:
        location = reverse((lat, lon), language="de")
        if location is None:
            return None
        return location.address
    except Exception as e:
        print(f"Reverse geocoding failed for ({lat}, {lon}): {e}")
        return None

addr_col = "address"

# identify rows with missing address
mask_missing_addr = (
    schools_final[addr_col].isna()
    | (schools_final[addr_col].astype(str).str.strip() == "")
)
print("Rows with missing address before Nominatim:", mask_missing_addr.sum())

# prepare a column to track if address was filled from Nominatim
if "address_from_nominatim" not in schools_final.columns:
    schools_final["address_from_nominatim"] = False
subset = schools_final[mask_missing_addr].copy()

subset["address_nominatim"] = subset.apply(
    lambda row: get_address_from_coords(row["latitude"], row["longitude"]),
    axis=1,
)
filled_mask = (
    mask_missing_addr
    & subset["address_nominatim"].notna()
)
schools_final.loc[subset.index[subset["address_nominatim"].notna()], addr_col] = \
    subset.loc[subset["address_nominatim"].notna(), "address_nominatim"]
schools_final.loc[subset.index[subset["address_nominatim"].notna()], "address_from_nominatim"] = True
print("Rows with missing address after Nominatim:", schools_final[addr_col].isna().sum())

Rows with missing address before Nominatim: 176
Rows with missing address after Nominatim: 0


In [25]:
# Fill missing school names
schools_final["name"] = schools_final["name"].astype("string").str.strip()

missing_name_mask = schools_final["name"].isna() | (schools_final["name"] == "")
missing_name_count = int(missing_name_mask.sum())

schools_final.loc[missing_name_mask, "name"] = "unknown_school_name"

print("Filled missing school names with placeholder:", missing_name_count)
print("Remaining missing names:", int(schools_final["name"].isna().sum()))

Filled missing school names with placeholder: 10
Remaining missing names: 0


## Summary

- OSM-based schools were cleaned, mapped to the common POI schema and spatially
  joined with LOR polygons to add `district`, `district_id`, `neighborhood`
  and `neighborhood_id`.
- Initially, 25 schools did not receive a `district_id`. They did not overlap
  with the records missing `name` and were located within the Berlin boundary.
  The spatial join was updated to use the centroid for polygon geometries
  (while keeping points as they are). After this change, all schools have a
  valid `district_id` (`schools_unified["district_id"].isna().sum() == 0`),
  so no records are excluded from the final export.
- Potential duplicates were identified using `(name, address, district)` and
  reduced to a single record per group in a deduplicated table
  (`schools_unified_dedup`), preferring more informative OSM features
  (e.g. ways with contact details).
- The final table `schools_final`:
  - has one row per school (after deduplication) and all rows have a valid `district_id`,
  - drops transformation- and source-specific helper columns,
  - keeps the POI core fields (id, coordinates, district, neighbourhood) and
    the main schools attributes (address, ownership, basic contact fields).
- Used reverse geocoding (Nominatim) to fill missing addresses from 
  latitude/longitude (rate-limited with delays/retry logic to respect the usage policy).
- Filled missing school names with the placeholder `Unknown school name`
  to avoid NULL values in the final export and keep all records.

In [26]:
# Save final dataset
output_path_final = "data/schools_unified_osm_only.csv"
schools_final.to_csv(output_path_final, index=False)
print(f"Saved final dataset with {len(schools_final)} rows to {output_path_final}")

Saved final dataset with 1017 rows to data/schools_unified_osm_only.csv


In [27]:
import os
from getpass import getpass

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


In [32]:
DB_USER = input("maximilian_burkhardt").strip()
DB_PASS = getpass("om2rkeJQeH4T4d4y8")

DB_HOST = "localhost"
DB_PORT = 5433

DB_NAME = input("layereddb").strip()

TARGET_SCHEMA = "berlin_source_data"
TARGET_TABLE  = f"schools_{DB_USER}"

url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASS,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

In [34]:
engine = create_engine(url, pool_pre_ping=True)

In [35]:
with engine.connect() as conn:
    print("connected:", conn.execute(text("SELECT 1")).scalar())
    print("db:", conn.execute(text("SELECT current_database()")).scalar())

connected: 1
db: layereddb


In [37]:
df_upload = schools_unified.copy()

if "geometry" in df_upload.columns:
    df_upload["geometry_wkt"] = df_upload["geometry"].astype(str)
    df_upload = df_upload.drop(columns=["geometry"])

if "latitude" in df_upload.columns and "longitude" in df_upload.columns:
    df_upload["coordinates"] = df_upload["latitude"].astype(str) + "," + df_upload["longitude"].astype(str)

if "id" in df_upload.columns:
    df_upload["id"] = df_upload["id"].astype(str)

df_upload.head()

,id,district_id,name,latitude,longitude,neighborhood,district,neighborhood_id,source,source_layer,...,last_source_update,last_ingested_at,is_active,lat,lon,has_district,has_neighborhood,source_type,geometry_wkt,coordinates
0,237838613,11007007,Rückert-Gymnasium,52.479969,13.338459,Schöneberg,Tempelhof-Schöneberg,0701,OSM,OSM_SCHOOLS,...,<NA>,<NA>,True,52.479969,13.338459,True,True,node,POINT (13.338459 52.479969),"52.4799688,13.3384592"
1,256912446,11004004,Erwin von Witzleben Grundschule,52.539441,13.288044,Charlottenburg-Nord,Charlottenburg-Wilmersdorf,0406,OSM,OSM_SCHOOLS,...,<NA>,<NA>,True,52.539441,13.288044,True,True,node,POINT (13.288044 52.539441),"52.53944109999999,13.288043699999998"
2,256913234,11004004,Sportschule im Olympiapark - Poelchau-Schule,52.521142,13.241634,Westend,Charlottenburg-Wilmersdorf,0405,OSM,OSM_SCHOOLS,...,<NA>,<NA>,True,52.521142,13.241634,True,True,node,POINT (13.241634 52.521142),"52.521142,13.241634199999998"
3,256913872,11004004,Anna Freud Schule,52.537659,13.288325,Charlottenburg-Nord,Charlottenburg-Wilmersdorf,0406,OSM,OSM_SCHOOLS,...,<NA>,<NA>,True,52.537659,13.288325,True,True,node,POINT (13.288326 52.537659),"52.5376587,13.2883255"
4,268915174,11004004,ABC-Kindergarten - Sabine Erdmann Vorschule,52.497835,13.301947,Wilmersdorf,Charlottenburg-Wilmersdorf,0402,OSM,OSM_SCHOOLS,...,<NA>,<NA>,True,52.497835,13.301947,True,True,node,POINT (13.301947 52.497835),"52.497834600000004,13.3019467"


In [ ]:
def preflight_report(df: pd.DataFrame):
    print("rows:", len(df))
    if "id" in df.columns:
        print("duplicate id:", df["id"].duplicated().sum())
        print("null id:", df["id"].isna().sum())

    for col in ["district_id", "neighborhood_id", "name"]:
        if col in df.columns:
            print(f"null {col}:", df[col].isna().sum())

    if "email" in df.columns:
        non_null = df["email"].dropna()
        print("duplicate email (non-null):", non_null.duplicated().sum())

    if "latitude" in df.columns and "longitude" in df.columns:
        print("lat min/max:", df["latitude"].min(), df["latitude"].max())
        print("lon min/max:", df["longitude"].min(), df["longitude"].max())
        

preflight_report(df_upload)

rows: 1072
duplicate id: 0
null id: 0
null district_id: 0
null neighborhood_id: 0
null name: 51
duplicate email (non-null): 2
lat min/max: 52.376033975618846 52.642552982133175
lon min/max: 13.127796103040813 13.730790393269407


In [40]:
create_schema_sql = f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA};"

create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {TARGET_SCHEMA}.{TARGET_TABLE} (
    id VARCHAR(50) PRIMARY KEY,
    district_id VARCHAR(20) NOT NULL,
    neighborhood_id VARCHAR(20),

    name VARCHAR(200) NOT NULL,

    school_type VARCHAR(100),
    operator VARCHAR(100),

    address VARCHAR(200),
    postal_code VARCHAR(10),
    phone VARCHAR(50),
    email VARCHAR(100),
    website VARCHAR(200),

    coordinates VARCHAR(200),
    latitude DECIMAL(9,6),
    longitude DECIMAL(9,6),

    neighborhood VARCHAR(100),
    district VARCHAR(100),

    opening_hours VARCHAR(200),

    geometry_wkt TEXT,

    CONSTRAINT district_id_fk
        FOREIGN KEY (district_id)
        REFERENCES {TARGET_SCHEMA}.districts(district_id)
        ON DELETE RESTRICT
        ON UPDATE CASCADE
);
"""

print("table ensured:", f"{TARGET_SCHEMA}.{TARGET_TABLE}")

table ensured: berlin_source_data.schools_maximilian_burkhardt


In [41]:
cols_sql = f"""
SELECT column_name
FROM information_schema.columns
WHERE table_schema = '{TARGET_SCHEMA}'
  AND table_name   = '{TARGET_TABLE}'
ORDER BY ordinal_position;
"""

with engine.connect() as conn:
    table_cols = [r[0] for r in conn.execute(text(cols_sql)).fetchall()]

missing_in_df = [c for c in table_cols if c not in df_upload.columns]
extra_in_df   = [c for c in df_upload.columns if c not in table_cols]

print("table cols:", table_cols)
print("missing in df:", missing_in_df)
print("extra in df:", extra_in_df)

df_insert = df_upload[[c for c in table_cols if c in df_upload.columns]].copy()

table cols: []
missing in df: []
extra in df: ['id', 'district_id', 'name', 'latitude', 'longitude', 'neighborhood', 'district', 'neighborhood_id', 'source', 'source_layer', 'external_id', 'primary_source', 'source_ids', 'school_name', 'school_type', 'school_subtype', 'grades_offered', 'is_primary', 'is_secondary', 'is_vocational', 'is_special_needs', 'languages', 'has_all_day_program', 'address', 'street', 'house_number', 'postal_code', 'city', 'ownership', 'official_school_id', 'is_barrier_free', 'accessibility_notes', 'phone', 'email', 'website', 'last_source_update', 'last_ingested_at', 'is_active', 'lat', 'lon', 'has_district', 'has_neighborhood', 'source_type', 'geometry_wkt', 'coordinates']


In [42]:
df_insert.to_sql(
    name=TARGET_TABLE,
    con=engine,
    schema=TARGET_SCHEMA,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print("insert done:", len(df_insert))

insert done: 1072


In [43]:
with engine.connect() as conn:
    cnt = conn.execute(text(f"SELECT COUNT(*) FROM {TARGET_SCHEMA}.{TARGET_TABLE};")).scalar()
    print("rowcount in db:", cnt)

    sample = conn.execute(text(f"""
        SELECT id, name, district_id, neighborhood_id, latitude, longitude
        FROM {TARGET_SCHEMA}.{TARGET_TABLE}
        ORDER BY id
        LIMIT 10;
    """)).fetchall()

sample

rowcount in db: 2


ProgrammingError: (psycopg2.errors.UndefinedColumn) column "id" does not exist
LINE 2:         SELECT id, name, district_id, neighborhood_id, latit...
                       ^

[SQL: 
        SELECT id, name, district_id, neighborhood_id, latitude, longitude
        FROM berlin_source_data.schools_maximilian_burkhardt
        ORDER BY id
        LIMIT 10;
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)